In [ ]:
import pandas as pd

df = pd.read_csv("data/survey_responses.csv", encoding="latin-1")
print(f"Loaded {len(df)} responses, {len(df.columns)} columns")
df.head()

In [ ]:
# identify multi-select columns (contain semicolons) and dummy-encode them
def dummy_encode_multiselect(df, columns):
    for col in columns:
        dummies = df[col].astype(str).str.get_dummies(sep=";")
        dummies.columns = [f"{col}_{opt.strip()}" for opt in dummies.columns]
        df = pd.concat([df, dummies], axis=1).drop(columns=[col])
    return df

df.columns = df.columns.str.strip()
multi_select_cols = [col for col in df.columns if df[col].astype(str).str.contains(";").any()]
df = dummy_encode_multiselect(df, multi_select_cols)
print(f"After dummy encoding: {len(df.columns)} columns")

In [ ]:
# rename columns to short Q-codes and drop admin/consent fields
rename_map = {
    # Q6 - hardest platform to report on
    "Which platform do you think makes it most difficult to report abusive or inappropriate behaviour? (Select all that apply)_About the same across all platforms": "Q6_same",
    "Which platform do you think makes it most difficult to report abusive or inappropriate behaviour? (Select all that apply)_Facebook": "Q6_facebook",
    "Which platform do you think makes it most difficult to report abusive or inappropriate behaviour? (Select all that apply)_Not sure": "Q6_not_sure",
    "Which platform do you think makes it most difficult to report abusive or inappropriate behaviour? (Select all that apply)_Snapchat": "Q6_snapchat",
    "Which platform do you think makes it most difficult to report abusive or inappropriate behaviour? (Select all that apply)_WhatsApp": "Q6_whatsapp",

    # Q8 - most likely to encounter grooming
    "On which platform do you think you are most likely to encounter grooming or abusive behaviour? (Select all that apply)_About the same across all platforms": "Q8_same",
    "On which platform do you think you are most likely to encounter grooming or abusive behaviour? (Select all that apply)_Facebook": "Q8_facebook",
    "On which platform do you think you are most likely to encounter grooming or abusive behaviour? (Select all that apply)_Snapchat": "Q8_snapchat",
    "On which platform do you think you are most likely to encounter grooming or abusive behaviour? (Select all that apply)_WhatsApp": "Q8_whatsapp",

    # Q10 - experienced uncomfortable contact
    "Have you ever received messages or content on any of these platforms that made you feel uncomfortable or unsafe? (Select all that apply)_No": "Q10_no",
    "Have you ever received messages or content on any of these platforms that made you feel uncomfortable or unsafe? (Select all that apply)_Prefer not to say": "Q10_prefer_not_to_say",
    "Have you ever received messages or content on any of these platforms that made you feel uncomfortable or unsafe? (Select all that apply)_Yes, on Facebook": "Q10_facebook",
    "Have you ever received messages or content on any of these platforms that made you feel uncomfortable or unsafe? (Select all that apply)_Yes, on Snapchat": "Q10_snapchat",
    "Have you ever received messages or content on any of these platforms that made you feel uncomfortable or unsafe? (Select all that apply)_Yes, on WhatsApp": "Q10_whatsapp",

    # Q14 - best protection
    "Overall, which platform(s) do you feel provides the best protection against grooming and online abuse?\xa0(Select all that apply)_Facebook": "Q14_facebook",
    "Overall, which platform(s) do you feel provides the best protection against grooming and online abuse?\xa0(Select all that apply)_None of them": "Q14_none",
    "Overall, which platform(s) do you feel provides the best protection against grooming and online abuse?\xa0(Select all that apply)_Not sure": "Q14_not_sure",
    "Overall, which platform(s) do you feel provides the best protection against grooming and online abuse?\xa0(Select all that apply)_WhatsApp": "Q14_whatsapp",

    # single-value questions
    "What is your age range?": "Q1_age",
    "What is your gender?": "Q2_gender",
    "What is your academic background?": "Q3_background",
    "How often do you use each of the following platforms?.Snapchat": "Q4_snapchat",
    "How often do you use each of the following platforms?.Facebook": "Q4_facebook",
    "How often do you use each of the following platforms?.WhatsApp": "Q4_whatsapp",
    "Were you aware that each of the following platforms can share or use your location data in some form?": "Q5_location_aware",
    "How confident are you that you could recognise signs of online grooming or manipulative or abusive behaviour on each of the following platforms?.Snapchat": "Q7_snapchat",
    "How confident are you that you could recognise signs of online grooming or manipulative or abusive behaviour on each of the following platforms?.Facebook": "Q7_facebook",
    "How confident are you that you could recognise signs of online grooming or manipulative or abusive behaviour on each of the following platforms?.WhatsApp": "Q7_whatsapp",
    "How often do you review or update your privacy settings on each of the following platforms?.Snapchat": "Q11_snapchat",
    "How often do you review or update your privacy settings on each of the following platforms?.Facebook": "Q11_facebook",
    "How often do you review or update your privacy settings on each of the following platforms?.WhatsApp": "Q11_whatsapp",
    "How safe do you feel using each of the following platforms for personal communication?.Snapchat": "Q12_snapchat",
    "How safe do you feel using each of the following platforms for personal communication?.Facebook": "Q12_facebook",
    "How safe do you feel using each of the following platforms for personal communication?.WhatsApp": "Q12_whatsapp",
    "How often do you accept contact or friend requests from people you do not know on each of the following platforms?.Snapchat": "Q13_snapchat",
    "How often do you accept contact or friend requests from people you do not know on each of the following platforms?.Facebook": "Q13_facebook",
    "How often do you accept contact or friend requests from people you do not know on each of the following platforms?.WhatsApp": "Q13_whatsapp",
    "Is there anything else you would like to share about your experience of safety or privacy on any of these platforms? (Optional)": "Q15_optional",
}

df = df.rename(columns=rename_map)

# drop admin and consent columns
drop_cols = [c for c in df.columns if c.startswith(("Id", "Start time", "Completion time", "Email", "Name", "I confirm", "I understand", "I agree"))]
df = df.drop(columns=drop_cols)

print(f"{len(df.columns)} columns after renaming")
sorted(df.columns.tolist())

In [ ]:
# convert Likert text responses to numeric scores
likert_map = {
    "Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Always": 5,
    "Not confident at all": 1, "Not very confident": 2, "Neutral": 3, "Confident": 4, "Very confident": 5,
    "Not safe at all": 1, "Not very safe": 2, "Safe": 4, "Very safe": 5,
    "Every day": 5, "Few times a week": 4, "Once a week or less": 3,
    "Yes, I was aware of this": 1, "No, I was not aware of this": 0,
}

likert_cols = [
    "Q4_snapchat", "Q4_facebook", "Q4_whatsapp",
    "Q5_location_aware",
    "Q7_snapchat", "Q7_facebook", "Q7_whatsapp",
    "Q11_snapchat", "Q11_facebook", "Q11_whatsapp",
    "Q12_snapchat", "Q12_facebook", "Q12_whatsapp",
    "Q13_snapchat", "Q13_facebook", "Q13_whatsapp",
]

for col in likert_cols:
    df[col] = df[col].map(likert_map)

df[likert_cols].head()

In [ ]:
# compute composite scores from the numeric survey data

# platform awareness (Q5 + Q7)
df["platform_awareness_score"] = (
    df["Q5_location_aware"] + df["Q7_snapchat"] + df["Q7_facebook"] + df["Q7_whatsapp"]
) / 4

# privacy behaviour (Q11 + Q13)
df["privacy_behaviour_score"] = (
    df["Q11_snapchat"] + df["Q11_facebook"] + df["Q11_whatsapp"] +
    df["Q13_snapchat"] + df["Q13_facebook"] + df["Q13_whatsapp"]
) / 6

# vulnerability exposure (Q10 + Q13)
df["vulnerability_exposure_score"] = (
    df["Q10_snapchat"] + df["Q10_facebook"] + df["Q10_whatsapp"] +
    df["Q13_snapchat"] + df["Q13_facebook"] + df["Q13_whatsapp"]
) / 6

# trust-risk gap per platform (Q12 safety - Q13 stranger contact)
df["trust_risk_gap_snapchat"] = df["Q12_snapchat"] - df["Q13_snapchat"]
df["trust_risk_gap_facebook"] = df["Q12_facebook"] - df["Q13_facebook"]
df["trust_risk_gap_whatsapp"] = df["Q12_whatsapp"] - df["Q13_whatsapp"]
df["trust_risk_gap_mean"] = (
    df["trust_risk_gap_snapchat"] + df["trust_risk_gap_facebook"] + df["trust_risk_gap_whatsapp"]
) / 3

# platform comparison (Q6 + Q8 + Q14)
df["platform_comparison_score"] = (
    df["Q6_snapchat"] + df["Q6_facebook"] + df["Q6_whatsapp"] +
    df["Q8_snapchat"] + df["Q8_facebook"] + df["Q8_whatsapp"] +
    df["Q14_facebook"] + df["Q14_whatsapp"]
) / 8

df[["platform_awareness_score", "privacy_behaviour_score",
    "vulnerability_exposure_score", "trust_risk_gap_mean",
    "platform_comparison_score"]].describe()

## Visualiser
Charts for the trust-risk gap analysis across Snapchat, Facebook and WhatsApp.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

PLATFORM_GAP_COLS = {
    "Snapchat": "trust_risk_gap_snapchat",
    "Facebook": "trust_risk_gap_facebook",
    "WhatsApp": "trust_risk_gap_whatsapp",
}

COMPOSITE_COLS = [
    "platform_awareness_score",
    "privacy_behaviour_score",
    "vulnerability_exposure_score",
    "trust_risk_gap_mean",
    "platform_comparison_score",
]

COMPOSITE_LABELS = {
    "platform_awareness_score":      "Platform\nAwareness",
    "privacy_behaviour_score":       "Privacy\nBehaviour",
    "vulnerability_exposure_score":  "Vulnerability\nExposure",
    "trust_risk_gap_mean":           "Trust-Risk\nGap (Mean)",
    "platform_comparison_score":     "Platform\nComparison",
}

PALETTE = {"Snapchat": "#FFFC00", "Facebook": "#1877F2", "WhatsApp": "#25D366"}
TECH_PALETTE = {"Technical": "#2E86AB", "Non-Technical": "#E84855"}

def bar_label(ax, bars, fmt="{:.2f}"):
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                    fmt.format(h), ha="center", va="bottom", fontsize=8)

# classify users as technical or non-technical based on Q3
tech_labels = {"Computer Science", "IT", "Cyber Security"}
df["_tech_group"] = df["Q3_background"].apply(
    lambda v: "Technical" if any(t.lower() in str(v).lower() for t in tech_labels)
    else "Non-Technical"
)
print(df["_tech_group"].value_counts())

In [ ]:
# Chart 1: Mean trust-risk gap by platform
means = {p: df[c].mean() for p, c in PLATFORM_GAP_COLS.items()}
sems = {p: df[c].sem() for p, c in PLATFORM_GAP_COLS.items()}

platforms = list(means.keys())
values = [means[p] for p in platforms]
errors = [sems[p] for p in platforms]
colours = [PALETTE[p] for p in platforms]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    platforms, values, color=colours, edgecolor="black",
    linewidth=0.8, yerr=errors, capsize=5,
    error_kw={"elinewidth": 1.2, "ecolor": "black"},
)
bar_label(ax, bars)
ax.set_title("Mean Trust-Risk Gap Score by Platform", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Platform", fontsize=11)
ax.set_ylabel("Mean Trust-Risk Gap Score", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.set_ylim(0, max(values) * 1.25 if max(values) > 0 else 1)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.savefig("figures/chart1_trust_risk_gap_by_platform.png", dpi=150)
plt.show()
print("Saved -> chart1_trust_risk_gap_by_platform.png")

In [ ]:
# Chart 2: Correlation heatmap across all five composite scores
available = [c for c in COMPOSITE_COLS if c in df.columns]
corr = df[available].corr()
labels = [COMPOSITE_LABELS[c] for c in available]

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5, linecolor="white",
    square=True, xticklabels=labels, yticklabels=labels,
    ax=ax, annot_kws={"size": 9},
)
ax.set_title("Correlation Matrix of Composite Scores", fontsize=14, fontweight="bold", pad=14)
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
fig.tight_layout()
plt.savefig("figures/chart2_correlation_heatmap.png", dpi=150)
plt.show()
print("Saved -> chart2_correlation_heatmap.png")

In [ ]:
# Chart 3: Composite scores by technical vs non-technical background
available = [c for c in COMPOSITE_COLS if c in df.columns]
groups = ["Technical", "Non-Technical"]
group_data = {g: df.loc[df["_tech_group"] == g, available].mean() for g in groups}
group_sem = {g: df.loc[df["_tech_group"] == g, available].sem() for g in groups}

n = len(available)
x = np.arange(n)
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
for i, group in enumerate(groups):
    offset = (i - 0.5) * width
    bars = ax.bar(
        x + offset,
        [group_data[group][c] for c in available],
        width, label=group,
        color=TECH_PALETTE[group], edgecolor="black",
        linewidth=0.7,
        yerr=[group_sem[group][c] for c in available],
        capsize=4, error_kw={"elinewidth": 1.1, "ecolor": "black"},
        alpha=0.88,
    )
    bar_label(ax, bars)

ax.set_title("Composite Scores: Technical vs Non-Technical Users", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Composite Score", fontsize=11)
ax.set_ylabel("Mean Score", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([COMPOSITE_LABELS[c] for c in available], fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Background", fontsize=10, title_fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

for g in groups:
    n_g = (df["_tech_group"] == g).sum()
    print(f"  {g}: n={n_g}")

fig.tight_layout()
plt.savefig("figures/chart3_composite_by_background.png", dpi=150)
plt.show()
print("Saved -> chart3_composite_by_background.png")

In [ ]:
# Chart 4: Mean Q12 (trust) vs Q13 (stranger contact) per platform
platforms = ["Snapchat", "Facebook", "WhatsApp"]
q12_cols = ["Q12_snapchat", "Q12_facebook", "Q12_whatsapp"]
q13_cols = ["Q13_snapchat", "Q13_facebook", "Q13_whatsapp"]

q12_means = [df[c].mean() for c in q12_cols]
q13_means = [df[c].mean() for c in q13_cols]
q12_sems = [df[c].sem() for c in q12_cols]
q13_sems = [df[c].sem() for c in q13_cols]

x = np.arange(len(platforms))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(
    x - width / 2, q12_means, width, label="Q12 – Perceived Safety (Trust)",
    color="#4C9BE8", edgecolor="black", linewidth=0.7,
    yerr=q12_sems, capsize=5, error_kw={"elinewidth": 1.2, "ecolor": "black"},
)
bars2 = ax.bar(
    x + width / 2, q13_means, width, label="Q13 – Stranger Contact (Risk)",
    color="#E85C5C", edgecolor="black", linewidth=0.7,
    yerr=q13_sems, capsize=5, error_kw={"elinewidth": 1.2, "ecolor": "black"},
)
bar_label(ax, bars1)
bar_label(ax, bars2)

ax.set_title("Perceived Safety vs Stranger Contact by Platform", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Platform", fontsize=11)
ax.set_ylabel("Mean Likert Score", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(platforms, fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.savefig("figures/chart4_trust_vs_stranger_contact.png", dpi=150)
plt.show()
print("Saved -> chart4_trust_vs_stranger_contact.png")

In [ ]:
# Chart 5: Composite scores by gender
groups = df["Q2_gender"].dropna().unique().tolist()
available = [c for c in COMPOSITE_COLS if c in df.columns]
group_data = {g: df.loc[df["Q2_gender"] == g, available].mean() for g in groups}
group_sem = {g: df.loc[df["Q2_gender"] == g, available].sem() for g in groups}

n = len(available)
x = np.arange(n)
width = 0.35
gender_colours = {"Male": "#5B8DB8", "Female": "#D4726A"}

fig, ax = plt.subplots(figsize=(11, 6))
for i, group in enumerate(groups):
    offset = (i - 0.5) * width
    bars = ax.bar(
        x + offset, [group_data[group][c] for c in available],
        width, label=group,
        color=gender_colours.get(group, "#999999"), edgecolor="black",
        linewidth=0.7,
        yerr=[group_sem[group][c] for c in available],
        capsize=4, error_kw={"elinewidth": 1.1, "ecolor": "black"},
        alpha=0.88,
    )
    bar_label(ax, bars)

ax.set_title("Composite Scores by Gender", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Composite Score", fontsize=11)
ax.set_ylabel("Mean Score", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([COMPOSITE_LABELS[c] for c in available], fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Gender", fontsize=10, title_fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

for g in groups:
    print(f"  {g}: n={(df["Q2_gender"] == g).sum()}")

fig.tight_layout()
plt.savefig("figures/chart5_composite_by_gender.png", dpi=150)
plt.show()
print("Saved -> chart5_composite_by_gender.png")

In [ ]:
# Chart 6: Composite scores by age band
def age_band(val):
    if val in ("18-20",):
        return "18-20"
    elif val in ("21-23",):
        return "21-23"
    else:
        return "24+"

df["_age_band"] = df["Q1_age"].apply(age_band)
age_order = ["18-20", "21-23", "24+"]
available = [c for c in COMPOSITE_COLS if c in df.columns]
group_data = {g: df.loc[df["_age_band"] == g, available].mean() for g in age_order}
group_sem = {g: df.loc[df["_age_band"] == g, available].sem() for g in age_order}

n = len(available)
x = np.arange(n)
width = 0.25
age_colours = {"18-20": "#7FB069", "21-23": "#E6AA68", "24+": "#C44536"}

fig, ax = plt.subplots(figsize=(11, 6))
for i, group in enumerate(age_order):
    offset = (i - 1) * width
    bars = ax.bar(
        x + offset, [group_data[group][c] for c in available],
        width, label=group,
        color=age_colours[group], edgecolor="black",
        linewidth=0.7,
        yerr=[group_sem[group][c] for c in available],
        capsize=4, error_kw={"elinewidth": 1.1, "ecolor": "black"},
        alpha=0.88,
    )
    bar_label(ax, bars)

ax.set_title("Composite Scores by Age Band", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Composite Score", fontsize=11)
ax.set_ylabel("Mean Score", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([COMPOSITE_LABELS[c] for c in available], fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Age Band", fontsize=10, title_fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

for g in age_order:
    print(f"  {g}: n={(df["_age_band"] == g).sum()}")

fig.tight_layout()
plt.savefig("figures/chart6_composite_by_age.png", dpi=150)
plt.show()
print("Saved -> chart6_composite_by_age.png")